# Random Forest Model — Low / High / All MP Datasets
Follows the same workflow as `model_development.ipynb` (LightGBM reference).
Feature selection uses RF-RFE; HPO uses BayesSearchCV (20 trials) + default trial 0.

In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Real, Integer
import joblib
import matplotlib.pyplot as plt

import feature_engineering_helper as hf

## 0. Configuration

In [2]:
### change to 1 for quick test, 0 for full run
quick_test = 0
### change to 1 for quick test, 0 for full run

if quick_test == 1:
    n_features_to_select = 40
    step                 = 50
    print('Quick test mode')
else:
    n_features_to_select = 1
    step                 = 5
    print('Full run mode')

data_prefix   = '../0_data/processed_data/'
figure_prefix = '../Figures/'

model_types = ['RF']
label       = 'MP_label'
non_feature_cols = ['SMILES', 'MP', 'Type'] + [label]
data_types  = ['L', 'H', 'All']

# MP threshold — consistent across all models in this project
MP_THRESHOLD = 250   # °C boundary between Low ('L') and High ('H')

Full run mode


## 1. Feature Selection (RFE with RF)

In [3]:
if quick_test == 1:
    df_all_feature        = pd.read_parquet(data_prefix + 'data_with_all_features_scaled.parquet').head(500)
    df_all_feature_scaled = df_all_feature
else:
    df_all_feature        = pd.read_parquet(data_prefix + 'data_with_all_features.parquet')
    df_all_feature_scaled = pd.read_parquet(data_prefix + 'data_with_all_features_scaled.parquet')

df_all_feature

,MACCS_0,MACCS_1,MACCS_10,MACCS_100,MACCS_101,MACCS_102,MACCS_103,MACCS_104,MACCS_105,MACCS_106,...,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,SMILES,Type
0,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0.399047,COc1ccc(cc1)C1(C)CCc2c(-c3c1cc(o3)C)c1c(o2)ccc...,Train
1,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0.497846,C[C@H]1[C@@H]2CC[C@@H]3[C@](C1=O)(C2)C(=O)OC[C...,Train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.667560,Cc1cc(Br)c(cc1Br)C,Train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.669548,OC(=O)c1ccc(c(c1)F)C,Train
4,0,0,0,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0.703933,OC(=O)C1CC(=O)c2c1cccc2,Train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17215,0,0,0,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0.438791,CN(CCNCc1cc(ccc1O)[N+](=O)[O-])C,Test
17216,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.490747,C[Si](C#Cc1ccc(cc1)C#C[Si](C)(C)C)(C)C,Test
17217,0,0,0,0,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0.579542,Brc1ccc(c(c1)C(F)(F)F)[N+](=O)[O-],Test
17218,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0.701361,OC(=O)C(C(=O)O)Cc1ccccc1,Test


In [4]:
def feature_engineering_workflow(model_type, df):
    data      = df.copy()
    tolerance = 0.01

    all_feature_cols = data.drop(columns=non_feature_cols, errors='ignore').columns.tolist()
    print(f'Total number of features: {len(all_feature_cols)}')
    print()

    # Variance filter
    variance_threshold = 0.01
    print(f'Variance threshold feature selection: variance_threshold={variance_threshold}')
    df_X_variance = hf.reduce_features_by_variance(data[all_feature_cols],
                                                    variance_threshold=variance_threshold)
    print()

    # RFE
    print(f'RFE feature selection: model={model_type}, tolerance={tolerance}, '
          f'n_features_to_select={n_features_to_select}, step={step}')
    RFE_results = hf.reduce_features_by_RFE(
        df_X_variance, data['MP'],
        model=model_type, tolerance=tolerance,
        n_features_to_select=n_features_to_select, step=step,
        metric='rmse', cv_strategy=None
    )
    print()

    hf.RFE_plot(RFE_results, tolerance, model_type,
                save_path=figure_prefix + f'RFE_plot_{model_type}.png')

    return df_X_variance, RFE_results, df_X_variance[RFE_results['best_features']]

In [ ]:
feature_engineering_dict = {}

for model_type in model_types:
    for data_type in data_types:

        data_train = df_all_feature[df_all_feature['Type'] == 'Train']
        if data_type == 'L':
            data_with_features_train = data_train[data_train[label] == 'L']
        elif data_type == 'H':
            data_with_features_train = data_train[data_train[label] == 'H']
        else:
            data_with_features_train = data_train

        print(f'Running feature engineering workflow with {model_type} model on {data_type} data')
        print(f'Number of samples: {data_with_features_train.shape[0]}, '
              f'Number of features: {data_with_features_train.shape[1]}')

        df_X_variance, RFE_results, df_X_RFE = feature_engineering_workflow(
            model_type, data_with_features_train
        )
        feature_engineering_dict[(model_type, data_type)] = RFE_results
        print()

with open('feature_engineering_dict_RF.pkl', 'wb') as f:
    pickle.dump(feature_engineering_dict, f)

Running feature engineering workflow with RF model on L data
Number of samples: 11439, Number of features: 388
Total number of features: 384

Variance threshold feature selection: variance_threshold=0.01
Original features: 384
Removed features: 50
Remaining features: 334

RFE feature selection: model=RF, tolerance=0.01, n_features_to_select=1, step=5


RFE Feature Selection:   1%|▏         | 1/67 iteration

Iteration 0/67 | Features: 329 | RMSE: 35.0995 ± 1.0381 | Removed: [MACCS_18, MACCS_32, MACCS_55, MACCS_61, RDKit_EState_VSA11]


## 2. Save Feature-Selected Datasets

In [ ]:
# Load (in case of kernel restart)
with open('feature_engineering_dict_RF.pkl', 'rb') as f:
    feature_engineering_dict = pickle.load(f)

In [ ]:
for model_type in model_types:
    for data_type in data_types:

        best_features = feature_engineering_dict[(model_type, data_type)]['best_features']
        print(f'Best features for {model_type} model - {data_type} data ({len(best_features)} features):')
        print(best_features)
        print()

        data_selected_features        = df_all_feature[non_feature_cols + best_features]
        data_selected_features_scaled = df_all_feature_scaled[non_feature_cols + best_features]

        data_selected_features.to_parquet(
            data_prefix + f'data_with_selected_features_{model_type}_{data_type}.parquet'
        )
        data_selected_features_scaled.to_parquet(
            data_prefix + f'data_with_selected_features_{model_type}_{data_type}_scaled.parquet'
        )

## 3. Model Development (BayesSearchCV — mirrors LightGBM workflow)

In [ ]:
def model_development(data, non_feature_cols, model_type, trials):

    X = data.drop(columns=non_feature_cols, errors='ignore')
    y = data['MP'].values
    strat_labels = data[label].values

    # Precompute folds stratified on label (not continuous MP)
    skf   = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    folds = list(skf.split(X, strat_labels))

    # Helper: run 10-fold CV for a given model instance
    def run_cv(model_instance):
        fold_rmses = []
        for train_idx, val_idx in folds:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            model_instance.fit(X_train, y_train)
            preds = model_instance.predict(X_val)
            fold_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
        return fold_rmses

    # RF model + search space
    # NOTE: RF does not require scaled input; unscaled parquet is used below
    if model_type == 'RF':
        default_model = RandomForestRegressor(random_state=42, n_jobs=-1)
        base_model    = RandomForestRegressor(random_state=42, n_jobs=-1)
        search_space  = {
            'n_estimators':      Integer(20,  500),
            'max_depth':         Integer(3,    12),
            'min_samples_split': Integer(2,    10),
            'min_samples_leaf':  Integer(1,    10),
            'max_features':      Real(0.1, 1.0),
        }
    else:
        raise ValueError(f"model_type must be 'RF'; got '{model_type}'")

    # Trial 0: default hyperparameters
    trial_results = {}
    fold_rmses_0  = run_cv(default_model)
    mean_0 = float(np.mean(fold_rmses_0))
    std_0  = float(np.std(fold_rmses_0))
    trial_results[0] = {'fold_rmses': fold_rmses_0, 'mean_rmse': mean_0, 'std_rmse': std_0}
    print(f'Trial  0 (default) | mean RMSE: {mean_0:.4f} ± {std_0:.4f}')

    # Trials 1-N: BayesSearchCV
    opt = BayesSearchCV(
        base_model,
        search_space,
        n_iter=trials,
        cv=folds,
        scoring='neg_root_mean_squared_error',
        random_state=42,
        n_jobs=1,
        refit=True,
    )
    opt.fit(X, y)

    n_splits = len(folds)
    for i in range(trials):
        fold_rmses = [-opt.cv_results_[f'split{s}_test_score'][i] for s in range(n_splits)]
        mean_rmse  = float(np.mean(fold_rmses))
        std_rmse   = float(np.std(fold_rmses))
        trial_results[i + 1] = {
            'fold_rmses': fold_rmses,
            'mean_rmse':  mean_rmse,
            'std_rmse':   std_rmse,
        }
        print(f'Trial {i+1:>2d} | mean RMSE: {mean_rmse:.4f} ± {std_rmse:.4f}')

    return trial_results, opt.best_estimator_

In [ ]:
for model in model_types:
    for data_type in data_types:

        # RF does not require feature scaling — load unscaled parquet
        data = pd.read_parquet(
            data_prefix + f'data_with_selected_features_{model}_{data_type}.parquet'
        )
        data = data[data['Type'] == 'Train'].reset_index(drop=True)

        if data_type != 'All':
            data = data[data[label] == data_type].reset_index(drop=True)

        print(f'\n=== Model: {model} ===')
        print(f'=== Data Type: {data_type} ===')
        print(f'Dataset shape: {data.shape} '
              f'(n_samples={data.shape[0]}, '
              f'n_features={data.shape[1] - len(non_feature_cols)})')

        trial_results, best_model = model_development(
            data, non_feature_cols, model_type=model, trials=20
        )

        with open(f'model_development_results_{model}_{data_type}.pkl', 'wb') as f:
            pickle.dump(trial_results, f)
        joblib.dump(best_model, f'best_model_{model}_{data_type}.joblib', compress=3)

## 4. Performance Plot

In [ ]:
def plot_model_performance(model_development_results_dict):
    trials     = sorted(model_development_results_dict.keys())
    mean_rmses = np.array([model_development_results_dict[t]['mean_rmse'] for t in trials])
    std_rmses  = np.array([model_development_results_dict[t]['std_rmse']  for t in trials])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(trials, mean_rmses, marker='o', linewidth=1.5, color='steelblue', label='Mean RMSE')
    ax.fill_between(trials,
                    mean_rmses - std_rmses,
                    mean_rmses + std_rmses,
                    alpha=0.25, color='steelblue', label='± 1 std')
    ax.axvline(x=0, color='grey', linestyle='--', linewidth=1, label='Default HP (trial 0)')

    best_trial = trials[int(np.argmin(mean_rmses))]
    best_rmse  = float(min(mean_rmses))
    ax.scatter([best_trial], [best_rmse], color='red', zorder=5,
               label=f'Best (trial {best_trial}, RMSE={best_rmse:.4f})')

    ax.set_xlabel('HP Tuning Iteration', fontsize=12)
    ax.set_ylabel('RMSE', fontsize=12)
    ax.set_title('Model Performance vs. HP Tuning Iteration', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
for model in model_types:
    for data_type in data_types:

        print(f'\n=== Performance plot for {model} ({data_type}) ===')

        with open(f'model_development_results_{model}_{data_type}.pkl', 'rb') as f:
            model_development_results = pickle.load(f)

        plot_model_performance(model_development_results)

## 5. Test Evaluation — Overall + Low-MP + High-MP

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

def evaluate_on_test(model, df, best_features, label_col=label):
    """Return metrics dict and predictions dataframe for a given split."""
    X    = df[best_features]
    y    = df['MP'].values
    pred = model.predict(X)

    out_df = pd.DataFrame({
        'SMILES'   : df['SMILES'].values,
        label_col  : df[label_col].values,
        'exp_MP'   : y,
        'pred_MP'  : pred,
        'error'    : pred - y,
        'abs_error': np.abs(pred - y),
    })

    def _m(sub):
        if len(sub) == 0:
            return {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan, 'n': 0}
        return {
            'RMSE': float(np.sqrt(mean_squared_error(sub['exp_MP'], sub['pred_MP']))),
            'MAE' : float(mean_absolute_error(sub['exp_MP'], sub['pred_MP'])),
            'R2'  : float(r2_score(sub['exp_MP'], sub['pred_MP'])),
            'n'   : len(sub),
        }

    cat = out_df[label_col].str.strip().str.upper()
    return out_df, _m(out_df), _m(out_df[cat == 'L']), _m(out_df[cat == 'H'])

In [ ]:
all_test_metrics = []

for model in model_types:
    for data_type in data_types:

        best_features = feature_engineering_dict[(model, data_type)]['best_features']

        # RF: unscaled test data
        df_sel  = pd.read_parquet(
            data_prefix + f'data_with_selected_features_{model}_{data_type}.parquet'
        )
        df_test = df_sel[df_sel['Type'] == 'Test']

        best_model = joblib.load(f'best_model_{model}_{data_type}.joblib')

        out_df, m_all, m_L, m_H = evaluate_on_test(best_model, df_test, best_features)

        print(f'\n=== {model} | train={data_type} ===')
        print(f'  [Overall]  RMSE={m_all["RMSE"]:.4f}  MAE={m_all["MAE"]:.4f}  R2={m_all["R2"]:.4f}  n={m_all["n"]}')
        print(f'  [Low-MP]   RMSE={m_L["RMSE"]:.4f}  MAE={m_L["MAE"]:.4f}  R2={m_L["R2"]:.4f}  n={m_L["n"]}')
        print(f'  [High-MP]  RMSE={m_H["RMSE"]:.4f}  MAE={m_H["MAE"]:.4f}  R2={m_H["R2"]:.4f}  n={m_H["n"]}')

        out_df.to_csv(f'test_predictions_{model}_{data_type}.csv', index=False)

        all_test_metrics.append({
            'model': model, 'train_data': data_type,
            **{f'overall_{k}': v for k, v in m_all.items()},
            **{f'low_{k}': v    for k, v in m_L.items()},
            **{f'high_{k}': v   for k, v in m_H.items()},
        })

pd.DataFrame(all_test_metrics).to_csv('rf_test_metrics_summary.csv', index=False)
print('\n✅ Done.')
print(pd.DataFrame(all_test_metrics).to_string(index=False))